In [1]:
import os
import sys
import shutil
import subprocess
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 0. Biopython
# ============================================================

if importlib.util.find_spec("Bio") is None:
    print("Biopython not found. Installing into current Jupyter kernel...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "biopython"])

from Bio import Entrez, SeqIO


# ============================================================
# 1. CONFIG
# ============================================================

NCBI_EMAIL = "your_email@example.com"  # replace if needed

OUTDIR = Path("mtDNA15_MAFFT_distances")
OUTDIR.mkdir(exist_ok=True)

RAW_FASTA = OUTDIR / "mtDNA15_raw.fasta"
CLEAN_FASTA = OUTDIR / "mtDNA15_clean.fasta"
ALIGNED_FASTA = OUTDIR / "mtDNA15_aligned_mafft.fasta"
MAFFT_LOG = OUTDIR / "mafft.stderr.log"

OUT_LABEL_MAP = OUTDIR / "T_labels_mapping.csv"

OUT_PDIST_T = OUTDIR / "Dref_MAFFT_pairwise_deletion_pdistance_Tlabels.csv"
OUT_JC69_T = OUTDIR / "Dref_MAFFT_pairwise_deletion_JC69_Tlabels.csv"
OUT_K80_T = OUTDIR / "Dref_MAFFT_pairwise_deletion_K80_Tlabels.csv"

OUT_PDIST_FULL = OUTDIR / "Dref_MAFFT_pairwise_deletion_pdistance_full_labels.csv"
OUT_JC69_FULL = OUTDIR / "Dref_MAFFT_pairwise_deletion_JC69_full_labels.csv"
OUT_K80_FULL = OUTDIR / "Dref_MAFFT_pairwise_deletion_K80_full_labels.csv"

# If mafft is not in PATH, set full path manually, e.g. "/usr/bin/mafft"
MAFFT_EXE = shutil.which("mafft") or "/usr/bin/mafft"

ACCESSIONS = {
    "KJ434962": "Allenopithecus_nigroviridis",
    "KT159932": "Cercocebus_atys",
    "KC757390": "Cercocebus_chrysogaster",
    "KJ434959": "Cercocebus_torquatus",
    "AY863426": "Cercopithecus_aethiops",
    "KC757391": "Cercopithecus_albogularis",
    "KJ434958": "Cercopithecus_diana",
    "KJ434957": "Cercopithecus_lhoesti",
    "KJ434956": "Cercopithecus_mitis",
    "MW160353": "Cercopithecus_neglectus",
    "KU682691": "Chlorocebus_aethiops_C1",
    "MN816163": "Chlorocebus_aethiops",
    "KU682693": "Chlorocebus_cynosuros_C3",
    "KM262190": "Chlorocebus_cynosuros",
    "KU682695": "Chlorocebus_djamdjamensis_C5",
}

FULL_LABELS = [f"{species}_{acc}" for acc, species in ACCESSIONS.items()]
T_LABELS = [f"T{i+1}" for i in range(len(FULL_LABELS))]


# ============================================================
# 2. HELPERS
# ============================================================

def run_command(cmd, stdout_path=None, stderr_path=None):
    print("\nRunning:")
    print(" ".join(map(str, cmd)))

    stdout_handle = open(stdout_path, "w", encoding="utf-8") if stdout_path else None
    stderr_handle = open(stderr_path, "w", encoding="utf-8") if stderr_path else None

    try:
        result = subprocess.run(
            list(map(str, cmd)),
            stdout=stdout_handle if stdout_handle else subprocess.PIPE,
            stderr=stderr_handle if stderr_handle else subprocess.PIPE,
            text=True,
            check=False,
        )

        if result.returncode != 0:
            if result.stderr:
                print(result.stderr)
            raise RuntimeError(f"Command failed with exit code {result.returncode}")

        if result.stdout and not stdout_path:
            print(result.stdout[:1000])

    finally:
        if stdout_handle:
            stdout_handle.close()
        if stderr_handle:
            stderr_handle.close()


def download_fasta(accessions, output_fasta):
    Entrez.email = NCBI_EMAIL
    ids = ",".join(accessions)

    print(f"Downloading {len(accessions)} sequences from NCBI...")
    with Entrez.efetch(
        db="nucleotide",
        id=ids,
        rettype="fasta",
        retmode="text",
    ) as handle:
        fasta_text = handle.read()

    output_fasta.write_text(fasta_text, encoding="utf-8")
    print(f"Saved raw FASTA: {output_fasta}")


def sanitize_fasta(raw_fasta, clean_fasta):
    records = list(SeqIO.parse(raw_fasta, "fasta"))
    if not records:
        raise ValueError("No records found in raw FASTA.")

    found = set()
    clean_records = []

    for rec in records:
        acc_full = rec.id
        acc = acc_full.split(".")[0]

        if acc not in ACCESSIONS:
            raise ValueError(f"Unexpected accession: {rec.id}")

        label = f"{ACCESSIONS[acc]}_{acc}"

        rec.id = label
        rec.name = label
        rec.description = ""

        clean_records.append(rec)
        found.add(acc)

    missing = set(ACCESSIONS) - found
    if missing:
        raise ValueError(f"Missing accessions in downloaded FASTA: {sorted(missing)}")

    # Reorder records exactly according to ACCESSIONS order
    by_id = {rec.id.split("_")[-1]: rec for rec in clean_records}
    ordered_records = [by_id[acc] for acc in ACCESSIONS.keys()]

    SeqIO.write(ordered_records, clean_fasta, "fasta")
    print(f"Saved clean FASTA: {clean_fasta}")


def read_alignment(aligned_fasta):
    records = list(SeqIO.parse(aligned_fasta, "fasta"))
    if not records:
        raise ValueError("No records found in aligned FASTA.")

    labels = [rec.id for rec in records]
    seqs = [str(rec.seq).upper() for rec in records]

    lengths = {len(s) for s in seqs}
    if len(lengths) != 1:
        raise ValueError(f"Aligned sequences have different lengths: {lengths}")

    return labels, seqs


def pairwise_deletion_counts(seq1, seq2):
    """
    Pairwise deletion:
    use only columns where both sequences have A/C/G/T.
    Gaps, N, ambiguity codes are excluded pairwise.
    """
    valid_bases = {"A", "C", "G", "T"}

    valid = 0
    mismatches = 0
    transitions = 0
    transversions = 0

    transition_pairs = {
        ("A", "G"), ("G", "A"),
        ("C", "T"), ("T", "C"),
    }

    for a, b in zip(seq1, seq2):
        if a not in valid_bases or b not in valid_bases:
            continue

        valid += 1

        if a != b:
            mismatches += 1

            if (a, b) in transition_pairs:
                transitions += 1
            else:
                transversions += 1

    return valid, mismatches, transitions, transversions


def jc69_distance(p):
    """
    Jukes-Cantor correction:
    d = -3/4 log(1 - 4p/3)
    """
    if not np.isfinite(p):
        return np.nan
    if p < 0:
        return np.nan
    if p == 0:
        return 0.0
    arg = 1.0 - 4.0 * p / 3.0
    if arg <= 0:
        return np.nan
    return float(-0.75 * np.log(arg))


def k80_distance(P, Q):
    """
    Kimura 2-parameter correction:
    P = transition proportion
    Q = transversion proportion
    d = -1/2 log(1 - 2P - Q) - 1/4 log(1 - 2Q)
    """
    if not np.isfinite(P) or not np.isfinite(Q):
        return np.nan

    a = 1.0 - 2.0 * P - Q
    b = 1.0 - 2.0 * Q

    if a <= 0 or b <= 0:
        return np.nan

    return float(-0.5 * np.log(a) - 0.25 * np.log(b))


def compute_distance_matrices(labels, seqs):
    n = len(seqs)

    D_p = np.zeros((n, n), dtype=float)
    D_jc = np.zeros((n, n), dtype=float)
    D_k80 = np.zeros((n, n), dtype=float)

    N_valid = np.zeros((n, n), dtype=int)
    N_mismatch = np.zeros((n, n), dtype=int)
    N_transition = np.zeros((n, n), dtype=int)
    N_transversion = np.zeros((n, n), dtype=int)

    for i in range(n):
        for j in range(i + 1, n):
            valid, mismatches, transitions, transversions = pairwise_deletion_counts(
                seqs[i], seqs[j]
            )

            N_valid[i, j] = N_valid[j, i] = valid
            N_mismatch[i, j] = N_mismatch[j, i] = mismatches
            N_transition[i, j] = N_transition[j, i] = transitions
            N_transversion[i, j] = N_transversion[j, i] = transversions

            if valid == 0:
                p = np.nan
                P = np.nan
                Q = np.nan
            else:
                p = mismatches / valid
                P = transitions / valid
                Q = transversions / valid

            D_p[i, j] = D_p[j, i] = p
            D_jc[i, j] = D_jc[j, i] = jc69_distance(p)
            D_k80[i, j] = D_k80[j, i] = k80_distance(P, Q)

    np.fill_diagonal(D_p, 0.0)
    np.fill_diagonal(D_jc, 0.0)
    np.fill_diagonal(D_k80, 0.0)

    counts = {
        "valid_sites": N_valid,
        "mismatches": N_mismatch,
        "transitions": N_transition,
        "transversions": N_transversion,
    }

    return D_p, D_jc, D_k80, counts


def save_matrix(M, labels, path):
    df = pd.DataFrame(M, index=labels, columns=labels)
    df.to_csv(path)
    print(f"Saved: {path}")


# ============================================================
# 3. DOWNLOAD / CLEAN FASTA
# ============================================================

if RAW_FASTA.exists():
    print(f"Raw FASTA already exists: {RAW_FASTA}")
else:
    download_fasta(list(ACCESSIONS.keys()), RAW_FASTA)

sanitize_fasta(RAW_FASTA, CLEAN_FASTA)


# ============================================================
# 4. MAFFT ALIGNMENT
# ============================================================

if not Path(MAFFT_EXE).exists() and shutil.which(MAFFT_EXE) is None:
    raise FileNotFoundError(
        f"Cannot find MAFFT executable: {MAFFT_EXE}\n"
        f"In WSL install it with: sudo apt install mafft"
    )

mafft_cmd = [
    MAFFT_EXE,
    "--auto",
    str(CLEAN_FASTA),
]

run_command(
    mafft_cmd,
    stdout_path=ALIGNED_FASTA,
    stderr_path=MAFFT_LOG,
)

print(f"Saved MAFFT alignment: {ALIGNED_FASTA}")


# ============================================================
# 5. COMPUTE DISTANCE MATRICES
# ============================================================

labels_alignment, seqs = read_alignment(ALIGNED_FASTA)

print("\nAlignment summary:")
print(f"  sequences: {len(seqs)}")
print(f"  columns:   {len(seqs[0])}")

print("\nAlignment labels:")
for i, lab in enumerate(labels_alignment, 1):
    print(f"{i:2d}. {lab}")

D_p, D_jc, D_k80, counts = compute_distance_matrices(labels_alignment, seqs)


# ============================================================
# 6. SAVE MATRICES
# ============================================================

# Save T-label mapping
pd.DataFrame({
    "T_label": T_LABELS,
    "full_label": FULL_LABELS,
}).to_csv(OUT_LABEL_MAP, index=False)
print(f"\nSaved: {OUT_LABEL_MAP}")

# T-label matrices for your completion pipeline
save_matrix(D_p, T_LABELS, OUT_PDIST_T)
save_matrix(D_jc, T_LABELS, OUT_JC69_T)
save_matrix(D_k80, T_LABELS, OUT_K80_T)

# Full-label matrices for manuscript transparency
save_matrix(D_p, FULL_LABELS, OUT_PDIST_FULL)
save_matrix(D_jc, FULL_LABELS, OUT_JC69_FULL)
save_matrix(D_k80, FULL_LABELS, OUT_K80_FULL)

# Optional: counts matrices
save_matrix(counts["valid_sites"], T_LABELS, OUTDIR / "pairwise_valid_sites_Tlabels.csv")
save_matrix(counts["mismatches"], T_LABELS, OUTDIR / "pairwise_mismatches_Tlabels.csv")
save_matrix(counts["transitions"], T_LABELS, OUTDIR / "pairwise_transitions_Tlabels.csv")
save_matrix(counts["transversions"], T_LABELS, OUTDIR / "pairwise_transversions_Tlabels.csv")


# ============================================================
# 7. QUICK DIAGNOSTICS
# ============================================================

def matrix_summary(name, M):
    off = M[np.triu_indices_from(M, k=1)]
    print(f"\n{name}")
    print(f"  min:    {np.nanmin(off):.6f}")
    print(f"  median: {np.nanmedian(off):.6f}")
    print(f"  mean:   {np.nanmean(off):.6f}")
    print(f"  max:    {np.nanmax(off):.6f}")
    print(f"  nan:    {np.isnan(off).sum()}")

matrix_summary("p-distance, pairwise deletion", D_p)
matrix_summary("JC69 corrected distance", D_jc)
matrix_summary("K80 corrected distance", D_k80)

valid_off = counts["valid_sites"][np.triu_indices_from(counts["valid_sites"], k=1)]
print("\nPairwise valid sites after gap/ambiguity deletion:")
print(f"  min:    {valid_off.min()}")
print(f"  median: {np.median(valid_off):.0f}")
print(f"  mean:   {valid_off.mean():.1f}")
print(f"  max:    {valid_off.max()}")

print("\nRecommended main D_ref file:")
print(f"  p-distance: {OUT_PDIST_T}")
print(f"  corrected:  {OUT_K80_T}")

Saved raw FASTA: mtDNA15_MAFFT_distances/mtDNA15_raw.fasta
Saved clean FASTA: mtDNA15_MAFFT_distances/mtDNA15_clean.fasta

Running:
/usr/bin/mafft --auto mtDNA15_MAFFT_distances/mtDNA15_clean.fasta
Saved MAFFT alignment: mtDNA15_MAFFT_distances/mtDNA15_aligned_mafft.fasta

Alignment summary:
  sequences: 15
  columns:   17095

Alignment labels:
 1. Allenopithecus_nigroviridis_KJ434962
 2. Cercocebus_atys_KT159932
 3. Cercocebus_chrysogaster_KC757390
 4. Cercocebus_torquatus_KJ434959
 5. Cercopithecus_aethiops_AY863426
 6. Cercopithecus_albogularis_KC757391
 7. Cercopithecus_diana_KJ434958
 8. Cercopithecus_lhoesti_KJ434957
 9. Cercopithecus_mitis_KJ434956
10. Cercopithecus_neglectus_MW160353
11. Chlorocebus_aethiops_C1_KU682691
12. Chlorocebus_aethiops_MN816163
13. Chlorocebus_cynosuros_C3_KU682693
14. Chlorocebus_cynosuros_KM262190
15. Chlorocebus_djamdjamensis_C5_KU682695

Saved: mtDNA15_MAFFT_distances/T_labels_mapping.csv
Saved: mtDNA15_MAFFT_distances/Dref_MAFFT_pairwise_deletion_